# 02. 비대칭 오류 비용을 반영한 scorer

목표: 실제 위반 미탐을 false alarm보다 7배 크게 보는 toy catalog 환경을 만듭니다. 아래 공식은 학습용이며 원문의 비공개 scorer를 재현한 것이 아닙니다.

In [ ]:
episodes = [
    {"id": "A", "is_violation": True},
    {"id": "B", "is_violation": False},
    {"id": "C", "is_violation": False},
    {"id": "D", "is_violation": True},
    {"id": "E", "is_violation": False},
    {"id": "F", "is_violation": True},
]

policies = {
    "allow_all": [False, False, False, False, False, False],
    "flag_all": [True, True, True, True, True, True],
    "candidate": [True, False, True, True, False, False],
    "oracle": [True, False, False, True, False, True],
}

assert all(len(predictions) == len(episodes) for predictions in policies.values())

In [ ]:
def evaluate_policy(episodes, predictions, missed_violation_cost=7.0, false_alarm_cost=1.0):
    confusion = {"tp": 0, "tn": 0, "fp": 0, "fn": 0}
    total_cost = 0.0

    for episode, predicted_violation in zip(episodes, predictions, strict=True):
        actual = episode["is_violation"]
        if actual and predicted_violation:
            confusion["tp"] += 1
        elif not actual and not predicted_violation:
            confusion["tn"] += 1
        elif not actual and predicted_violation:
            confusion["fp"] += 1
            total_cost += false_alarm_cost
        else:
            confusion["fn"] += 1
            total_cost += missed_violation_cost

    return {"cost": total_cost, **confusion}

results = {
    name: evaluate_policy(episodes, predictions)
    for name, predictions in policies.items()
}

for name, result in sorted(results.items(), key=lambda item: item[1]["cost"]):
    print(f"{name:10s} cost={result['cost']:4.1f} confusion={result}")

assert results["allow_all"]["cost"] == 21
assert results["flag_all"]["cost"] == 3
assert results["candidate"]["cost"] == 8
assert results["oracle"]["cost"] == 0

In [ ]:
# 예상 비용을 최소화하는 threshold를 유도합니다.
# violation으로 flag할 비용: (1-p) * false_alarm_cost
# allow할 비용: p * missed_violation_cost
def optimal_threshold(missed_violation_cost: float, false_alarm_cost: float) -> float:
    return false_alarm_cost / (missed_violation_cost + false_alarm_cost)

threshold = optimal_threshold(7.0, 1.0)
print(f"7:1 비용에서 이론적 violation threshold: {threshold:.3f}")
assert threshold == 0.125

## 실무 확장

최종 판정만 채점하지 말고 category 유효성, attribute 근거, tool-call 예산, human escalation과 정책 위반을 별도 지표로 기록하세요. 하나의 reward로 합치기 전 각 component metric을 확인해야 reward hacking을 발견하기 쉽습니다.